# 09 — Pilotdaten der Zielhardware

Auswertung der eigenen Aufnahmen (ESP32-C6 + MPU-6500, 100 Hz,
±16 g, ±2000 °/s) an Hüfte und beiden Knöcheln. Die Sohlenaufnahmen (L_Insole) werden
nicht verwendet (unvollständig, nur eine Seite).

| Teil | Frage |
|---|---|
| A | Zero-Shot: Wie gut erkennen Basis- und Lower-Limb-Modell Stürze? |
| B | Betrieb: Erkennung je Sturz und Fehlalarme mit der Alarmlogik der Firmware (3 von 5, 0,2 s) |
| C | Einbaulage: Wie viel ändert eine Ausrichtung an der Schwerkraft des Trainingsdatensatzes? |
| D | Nachtraining auf den Pilotdaten (dreifach, nach Aufnahmen getrennt) |

**Einschränkung:** eine Person. Teil D trennt nach Aufnahmen, nicht
nach Probanden, und ist daher optimistisch. Teile A–C sind gültig, da
kein Modell diese Daten gesehen hat.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow

import os, sys, shutil, time, json, pickle
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import urllib.request
_LIB_URL = "https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py"
_LIB_DRIVE = "/content/drive/MyDrive/ML_Fall_Detection/fall_lib.py"
try:
    urllib.request.urlretrieve(_LIB_URL, "/content/fall_lib.py")
    print("[fall_lib] von GitHub geladen")
except Exception as _e:
    print(f"[fall_lib] GitHub nicht erreichbar ({_e}) -- nutze Drive-Kopie")
    shutil.copy(_LIB_DRIVE, "/content/fall_lib.py")
sys.path.insert(0, "/content")
import importlib, fall_lib as fl
importlib.reload(fl)
for _f in ("load_pilot", "rotation_between", "voting_eval", "finetune"):
    assert hasattr(fl, _f), f"fall_lib veraltet ({_f} fehlt) -- neue Version auf Drive legen."
print("[fall_lib] Fensterlaenge", fl.WIN_LEN, "bei", fl.FS_TARGET, "Hz")

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
PILOT = f"{BASE}/Pilotdaten"
CACHE = Path(f"{BASE}/cache")

from tensorflow import keras
def lade_modell(pfad):
    m = keras.models.load_model(pfad, compile=False)
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

MODELLE = {"Basis": lade_modell(f"{BASE}/results/fall_model.keras"),
           "Lower-Limb": lade_modell(f"{BASE}/results/fall_model_lowerlimb.keras")}
_aug = Path(f"{BASE}/results/fall_model_lowerlimb_aug.keras")   # aus Notebook 11
if _aug.exists():
    MODELLE["LL-aug (B2r)"] = lade_modell(str(_aug))
run = fl.new_run("09_pilotdaten")
ergebnisse = {}

## Aufnahmen

In [ ]:
alle = [t for t in fl.load_pilot(PILOT) if "INSOLE" not in t["position"]]   # Sohle ausgeschlossen
POS = sorted({t["position"] for t in alle})
for p in POS:
    for f in sorted({t["datei"] for t in alle if t["position"] == p and t["label"]}):
        n = sum(1 for t in alle if t["position"] == p and t["datei"] == f)
        print(f"  {p:<12} {f:<34} {n} Aufprall(e)")

## Teil A — Zero-Shot (aufprallzentriertes Fenster)

In [ ]:
def vorhersage(m, X):
    return m.predict(fl.znorm(X), batch_size=512, verbose=0).ravel()

teilA = {}
print(f"{'Position':<13}{'Modell':<12}{'Se':>7}{'95%-KI':>17}{'Sp':>7}")
for p in POS:
    X, y, _, _ = fl.build_windows([t for t in alle if t["position"] == p], mode="standard")
    for name, m in MODELLE.items():
        pr = vorhersage(m, X)
        tp = int(((pr > .5) & (y == 1)).sum()); fn = int(((pr <= .5) & (y == 1)).sum())
        tn = int(((pr <= .5) & (y == 0)).sum()); fp = int(((pr > .5) & (y == 0)).sum())
        ci = fl.wilson_ci(tp, tp + fn)
        teilA[f"{p}|{name}"] = {"Se": tp / (tp + fn), "Se_ci95": ci, "Sp": tn / (tn + fp),
                                "TP": tp, "FN": fn, "FP": fp, "TN": tn}
        print(f"{p:<13}{name:<12}{tp/(tp+fn):>7.2f}   [{ci[0]:.2f}; {ci[1]:.2f}]{tn/(tn+fp):>7.2f}")
ergebnisse["teilA_zero_shot"] = teilA

## Teil B — Betrieb mit Alarmlogik

Wie in der Firmware: Inferenz alle 0,2 s, Alarm bei 3 positiven von 5
Entscheidungen (Schwelle 0,5). Ein Sturz gilt als erkannt, wenn bis
1 s nach dem Aufprall ein Alarm ausgelöst wird. Gezählt werden
Fehlalarm-**Ereignisse** in den Alltagsaufnahmen.

In [ ]:
SCHRITT = int(0.2 * fl.FS_TARGET)

def folgen(m, trials, R=None):
    seqs = []
    for t in trials:
        x = t["data"] if R is None else fl.rotate_windows(t["data"][None], R)[0]
        W = np.asarray(fl.window_sliding(x, fl.WIN_LEN, SCHRITT))
        if not len(W):
            continue
        p = vorhersage(m, W)
        if t["label"]:
            c = t["impact_idx"]
            lo = max(0, (c - fl.WIN_LEN) // SCHRITT)
            seqs.append((p[lo:c // SCHRITT + 6], True))
        else:
            seqs.append((p, False))
    return seqs

def betrieb(m, trials, R=None):
    r = fl.voting_eval(folgen(m, trials, R), 0.5, 3, 5, SCHRITT / fl.FS_TARGET)
    r1 = fl.voting_eval(folgen(m, trials, R), 0.5, 1, 1, SCHRITT / fl.FS_TARGET)
    return {"Se_3v5": r["Se"], "fp_3v5": r["fp_events"], "Se_1v1": r1["Se"],
            "fp_1v1": r1["fp_events"], "adl_min": round(r["adl_hours"] * 60, 1)}

# Feste Einbaulage der Firmware (modell.h, EINBAU_SOHLE): Sohle flach,
# Bauteilseite oben, USB-C zur Aussenseite -> Achsrahmen Knoechel
EINBAU = {p: np.array([[1, 0, 0], [0, 0, -1], [0, 1, 0]], float)
          for p in {t["position"] for t in alle} if "INSOLE" in p}

teilB = {}
print(f"{'Position':<13}{'Modell':<12}{'Se 1v1':>8}{'Se 3v5':>8}{'FP-Ereign. 3v5':>16}{'ADL min':>9}")
for p in POS:
    tr = [t for t in alle if t["position"] == p]
    for name, m in MODELLE.items():
        r = betrieb(m, tr); teilB[f"{p}|{name}"] = r
        print(f"{p:<13}{name:<12}{r['Se_1v1']:>8.2f}{r['Se_3v5']:>8.2f}{r['fp_3v5']:>16}{r['adl_min']:>9}")
        if p in EINBAU:
            r = betrieb(m, tr, EINBAU[p]); teilB[f"{p}+Einbau|{name}"] = r
            print(f"{p+'+Einbau':<13}{name:<12}{r['Se_1v1']:>8.2f}{r['Se_3v5']:>8.2f}{r['fp_3v5']:>16}{r['adl_min']:>9}")
ergebnisse["teilB_betrieb"] = teilB

## Teil C — Einbaulage

Jedes Modell erwartet die Achslage seines Trainingsdatensatzes. Hier
wird die mittlere Schwerkraftrichtung im Stehen auf die des
Trainingsdatensatzes gedreht (kleinste Drehung). Die Drehung um die
Schwerkraftachse bleibt dabei unbestimmt. Verbessert sich das Ergebnis
deutlich, erklärt die Einbaulage einen Teil der Domänenlücke.

Referenz Basis: SisFall-Alltagsaufnahmen. Referenz Lower-Limb:
CAPPIMU-Schienbein (Cache aus Notebook 10).

In [ ]:
REF = {}
sis_lokal = "/content/SisFall"
if not Path(sis_lokal).exists():
    shutil.copytree(f"{BASE}/SisFall_dataset", sis_lokal)
REF["Basis"] = fl.mean_gravity(fl.load_sisfall(sis_lokal, verbose=False))
knie = pickle.load(open(CACHE / "cappimu_knee_r.pkl", "rb")) + \
       pickle.load(open(CACHE / "cappimu_knee_l.pkl", "rb"))
REF["Lower-Limb"] = fl.mean_gravity(knie); del knie
print("Schwerkraft Referenz:", {k: np.round(v, 2).tolist() for k, v in REF.items()})

teilC = {}
print(f"\n{'Position':<13}{'Modell':<12}{'Winkel':>8}{'Se 3v5 vorher':>15}{'nachher':>9}{'FP vorher':>11}{'nachher':>9}")
for p in POS:
    tr = [t for t in alle if t["position"] == p]
    g = fl.mean_gravity(tr)
    for name, m in MODELLE.items():
        R = fl.rotation_between(g, REF[name])
        w = float(np.degrees(np.arccos(np.clip(g @ REF[name], -1, 1))))
        v, n = teilB[f"{p}|{name}"], betrieb(m, tr, R)
        teilC[f"{p}|{name}"] = {"winkel_grad": round(w), "nachher": n, "R": R.round(4).tolist()}
        print(f"{p:<13}{name:<12}{w:>7.0f}°{v['Se_3v5']:>15.2f}{n['Se_3v5']:>9.2f}"
              f"{v['fp_3v5']:>11}{n['fp_3v5']:>9}")
ergebnisse["teilC_einbaulage"] = teilC

## Teil C2 — Hüfte in den SisFall-Achsrahmen

SisFall (Gürtelschnalle): **+z vorwärts, +y in Schwerkraftrichtung (nach unten), +x nach rechts**
(Sucerquia et al. 2017). Ein Beschleunigungssensor misst im Stand +1 g auf der Achse, die nach **oben**
zeigt; in SisFall also y ≈ −1 g.

Eigener Prototyp an der Hüfte: im Stand x ≈ −0,9 g → **+x zeigt nach unten**. Liegelage nach
Vorwärtssturz z ≈ −0,9 g (Rücken oben), nach Rückwärtssturz z ≈ +0,8 g → **+z zeigt nach vorn**.
Damit ist rechts = unten × vorn = −y. Feste Drehung: x_S = −y, y_S = x, z_S = z, danach Ausgleich
der Neigung im Stand (Platine ca. 25° gekippt). Alles aus der Einbaulage, nicht aus den Ergebnissen.

In [ ]:
R_GEO = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]], float)

def sisfall_ausrichtung(trials):
    g = fl.mean_gravity(trials)                       # Stand, eigener Sensor
    # Kontrolle der Vorwaertsrichtung aus den Liegephasen
    for art in ("fall_forward", "fall_backward"):
        L = [t["data"][t["impact_idx"] + 75:t["impact_idx"] + 250, :3].mean(0)
             for t in trials if t["label"] and t["activity"] == art]
        if L:
            print(f"  Liegelage {art:<14} g = {np.round(np.mean(L, 0), 2)}")
    return fl.rotation_between(R_GEO @ g, np.array([0., -1., 0.])) @ R_GEO

tw = [t for t in alle if t["position"] == "WAIST"]
if tw:
    R_W = sisfall_ausrichtung(tw)
    print("R (Huefte -> SisFall):\n", np.round(R_W, 2))
    teilC2 = {}
    for name, R in [("roh", None), ("nur Achsen", R_GEO), ("Achsen + Neigung", R_W)]:
        r = fl.voting_eval(folgen(MODELLE["Basis"], tw, R), 0.5, 3, 5, 0.2)
        teilC2[name] = r
        print(f"  Basis {name:<18} Se {r['detected']}/{r['total_falls']}  FP-Ereignisse {r['fp_events']}")
    ergebnisse["teilC2_huefte_sisfall"] = teilC2

## Teil D — Nachtraining auf den Pilotdaten

Ausgangspunkt ist das Lower-Limb-Modell. Dreifach kreuzvalidiert: Die
Sturzdateien werden nach ihrer Nummer (00/01/02) und die
Alltagsaufnahmen nach Dritteln ihrer Dauer auf die Falten verteilt; je
Falte wird auf zwei Dritteln trainiert und auf dem dritten bewertet.
Positionen mit nur einer Sturzdatei je Sturzart werden nach Sturzart
aufgeteilt. Für die Sohle wird vorher die feste Einbaulage der Firmware
angewendet.

In [ ]:
def falten_zuordnen(trials, k=3):
    nummern = sorted({t["datei"][:2] for t in trials if t["label"]})
    nach_nummer = len(nummern) >= k
    arten = sorted({t["activity"] for t in trials if t["label"]})
    teile = []
    for t in trials:
        if t["label"]:
            g = (nummern.index(t["datei"][:2]) if nach_nummer
                 else arten.index(t["activity"])) % k
            teile.append((g, t))
        else:
            n = len(t["data"])
            for j in range(k):
                seg = t["data"][j * n // k:(j + 1) * n // k]
                if len(seg) >= fl.WIN_LEN:
                    teile.append((j, {**t, "data": seg}))
    return teile

teilD = {}
for p in POS:
    tr = [t for t in alle if t["position"] == p]
    if p in EINBAU:
        tr = [{**t, "data": fl.rotate_windows(t["data"][None], EINBAU[p])[0]} for t in tr]
    teile = falten_zuordnen(tr)
    seq_vor, seq_nach = [], []
    for g in range(3):
        train = [t for f, t in teile if f != g]; test = [t for f, t in teile if f == g]
        if not any(t["label"] for t in test):
            continue
        Xf = [fl.window_impact_centered(t["data"], impact_idx=t["impact_idx"])[0]
              for t in train if t["label"]]
        Xa = [w for t in train if not t["label"] for w in fl.window_sliding(t["data"], fl.WIN_LEN, 25)]
        X = np.asarray(Xf + Xa, np.float32); y = np.r_[np.ones(len(Xf)), np.zeros(len(Xa))]
        m = fl.finetune(MODELLE["Lower-Limb"], fl.znorm(X), y)
        seq_vor += folgen(MODELLE["Lower-Limb"], test); seq_nach += folgen(m, test)
    vor = fl.voting_eval(seq_vor, 0.5, 3, 5, 0.2); nach = fl.voting_eval(seq_nach, 0.5, 3, 5, 0.2)
    teilD[p] = {"vorher": vor, "nachher": nach}
    print(f"{p:<13} vorher Se {vor['Se']:.2f}, FP-Ereign. {vor['fp_events']:>3} | "
          f"nachher Se {nach['Se']:.2f}, FP-Ereign. {nach['fp_events']:>3}  "
          f"({vor['adl_hours']*60:.1f} min ADL)")
ergebnisse["teilD_nachtraining"] = teilD

## Speichern

In [ ]:
fl.save_results(run, ergebnisse, config=fl.default_config(),
    notes=("Pilotdaten: Zero-Shot, Betrieb 3v5, Einbaulage (Schwerkraft), "
           "SisFall-Achsen Huefte, Nachtraining 3-fach. Eine Person, ohne Sohle."))